In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras import backend as K
from tensorflow.keras.regularizers import l2

In [ ]:
# Laden des CIFAR-10-Datensatzes
(x_train0, y_train0), (x_test0, y_test0) = tf.keras.datasets.cifar10.load_data()

class_label = 1 # Klasse 1 ("automobile") filtern

# Trainingsdaten filtern
x_train = x_train0[y_train0.flatten() == class_label]
y_train = y_train0[y_train0.flatten() == class_label]

# Testdaten filtern
x_test = x_test0[y_test0.flatten() == class_label]
y_test = y_test0[y_test0.flatten() == class_label]

print(f'Gefilterte Trainingsdaten: {x_train.shape}')
print(f'Gefilterte Testdaten: {x_test.shape}')

# Normalisieren der Bilder auf den Bereich [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Bilder reshapen, um sie für den Autoencoder vorzubereiten
x_train = x_train.reshape((len(x_train), 32, 32, 3))
x_test = x_test.reshape((len(x_test), 32, 32, 3))

# In tf.data.Dataset konvertieren
train_dataset = tf.data.Dataset.from_tensor_slices(x_train).batch(64).shuffle(buffer_size=256)
test_dataset = tf.data.Dataset.from_tensor_slices(x_test).batch(64)

In [ ]:
def sampling(args):
    """Reparametrisierungstrick von VAE"""
    z_mean, z_log_var = args
    batch = tf.shape(z_mean)[0]
    dim = tf.shape(z_mean)[1]
    epsilon = tf.random.normal(shape=(batch, dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
class VAE(tf.keras.Model):
    def __init__(self, input_shape, latent_dim):
        super(VAE, self).__init__()
        self.encoder = self.build_encoder(input_shape, latent_dim)
        self.decoder = self.build_decoder(input_shape, latent_dim)
        
    def build_encoder(self, input_shape, latent_dim):
        encoder_input = layers.Input(shape=input_shape)
        x = layers.Flatten()(encoder_input)
        x = layers.Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)              
        x = layers.Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)              
        z_mean = layers.Dense(latent_dim, name='z_mean')(x)
        z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
        z = layers.Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])
        return models.Model(encoder_input, [z_mean, z_log_var, z], name='encoder')

    def build_decoder(self, input_shape, latent_dim):
        latent_inputs = layers.Input(shape=(latent_dim,), name='z_sampling')
        x = layers.Dense(64, activation='relu', kernel_regularizer=l2(0.001))(latent_inputs) 
        x = layers.Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)             
        x = layers.Dense(np.prod(input_shape), activation='sigmoid')(x)
        decoder_output = layers.Reshape(input_shape)(x)
        return models.Model(latent_inputs, decoder_output, name='decoder')
    
    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstructed = self.decoder(z)
        
        # Berechnung des Rekonstruktionsverlustes
        reconstruction_loss = tf.keras.losses.binary_crossentropy(inputs, reconstructed)
        reconstruction_loss = tf.reduce_mean(reconstruction_loss) * np.prod(inputs.shape[1:])
        
        # Berechnung des KL-Divergenz-Verlustes
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        
        # Kombinierter Verlust
        self.add_loss(reconstruction_loss + kl_loss)
        
        return reconstructed

In [ ]:
# VAE-Modell kompilieren
vae = VAE((32, 32, 3), latent_dim=2) #latent_dim=32
vae.compile(optimizer='adam') #tf.keras.optimizers.Adam(learning_rate=0.0005)

In [ ]:
# VAE trainieren
history = vae.fit(train_dataset, epochs=100, validation_data=test_dataset)

In [ ]:
def plot_reconstructions(model, dataset, n_images=10):
    dataset_iter = iter(dataset)
    batch = next(dataset_iter)
    reconstructions = model.predict(batch[:n_images])
    fig, axes = plt.subplots(2, n_images, figsize=(20, 4))
    for i in range(n_images):
        # Originalbild
        axes[0, i].imshow(batch[i])
        axes[0, i].axis('off')
        
        # Rekonstruiertes Bild
        axes[1, i].imshow(reconstructions[i])
        axes[1, i].axis('off')
    plt.show()

In [ ]:
# Rekonstruktionen anzeigen
plot_reconstructions(vae, test_dataset)

In [ ]:
# Plot the training and validation losses
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
def plot_latent_space(encoder, data, labels, num_samples=1000):
    """
    Visualisiert den latenten Raum eines Variational Autoencoders.
    
    :param encoder: Das Encoder-Modell
    :param data: Eingabedaten, die visualisiert werden sollen
    :param labels: Zu den Eingabedaten gehörende Labels
    :param num_samples: Anzahl der zu visualisierenden Stichproben
    """
    # Reduziere die Datenmenge für eine schnellere Visualisierung
    data = data[:num_samples]
    labels = labels[:num_samples]

    # Erhalte die latenten Variablen vom Encoder
    z_mean, _, _ = encoder.predict(data)

    # Erstelle den Scatter-Plot des latenten Raums
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(z_mean[:, 0], z_mean[:, 1], c=labels, cmap='viridis', alpha=0.5)
    plt.colorbar(scatter)
    plt.xlabel('Latente Dimension 1')
    plt.ylabel('Latente Dimension 2')
    plt.title('Latenter Raum des VAE')
    plt.show()


In [ ]:
plot_latent_space(vae.encoder, x_test, y_test)

In [ ]:
def plot_image_from_latent(decoder, z_sample):
    """
    Erzeugt und visualisiert ein Bild aus einem Punkt im latenten Raum.

    :param decoder: Das Decoder-Modell
    :param z_sample: Ein Punkt im latenten Raum
    """
    # Rekonstruiere das Bild aus dem latenten Punkt
    reconstructed_image = decoder.predict(np.array([z_sample]))

    # Zeige das rekonstruierte Bild
    plt.figure(figsize=(3, 3))
    plt.imshow(reconstructed_image[0, :, :, 0], cmap='gray')
    plt.axis('off')
    plt.title('Bild aus latenten Raum')
    plt.show()

In [ ]:
# Beispielpunkt aus dem latenten Raum (z.B. (0, 0))
z_point = [1, 1]
plot_image_from_latent(vae.decoder, z_point)